# Procesamiento de datos con APIs - SpaceX API

In [1]:
%pip install pandas matplotlib seaborn requests

  Using cached matplotlib-3.10.8-cp314-cp314-win_amd64.whl.metadata (52 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached requests-2.33.1-py3-none-any.whl.metadata (4.8 kB)
  Using cached tzdata-2026.1-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached contourpy-1.3.3-cp314-cp314-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.62.1-cp314-cp314-win_amd64.whl.metadata (119 kB)
  Using cached kiwisolver-1.5.0-cp314-cp314-win_amd64.whl.metadata (5.2 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   -------------------------------- ------- 8.1/9.9 MB 46.1 MB/s eta 0:00:01
   ---------------------------


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import json
import numpy as np

# Path permite construir rutas de archivos de forma más robusta que concatenar strings.
from pathlib import Path

from asyncio import sleep
# asyncio es una librería para programación asíncrona en Python

# Controla cuantas columnas se muestran (pandas)
pd.set_option('display.max_columns', 50)

# Controla el ancho de impresion en consola del notebook
pd.set_option('display.width', 200)

# Limita el largo de strings en cada celda mostrada (para facilitar lectura)
pd.set_option('display.max_colwidth', 80)

# Aplica un estilo visual global de seaborn a los gráficos. whitegrid es un estilo con cuadricula y fondo blanco
# palette='deep' es una paleta de colores predefinida de seaborn que se aplica a los gráficos para mejorar su apariencia visual
sns.set_theme(style="whitegrid", palette='deep')

# Define el tamaño por defecto de los gráficos generados con matplotlib.
# Esto afecta a todos los gráficos que se creen después de esta línea,
# a menos que se especifique un tamaño diferente para un gráfico específico.
plt.rcParams['figure.figsize'] = (10, 6)


BASE_URL = 'https://api.spacexdata.com/v4'


CACHE_DIR = Path('data/spacex_cache')


# Funcion para contruir la ruta del archivo cache de un endpoint
def cache_path(endpoint: str) -> Path:
    # Ejemplo: endpoint = 'launches' -> data/spacex_cache/launches.json
    return CACHE_DIR / f"{endpoint}.json"

# Funcion para cargar datos de un endpoint desde el cache local
def load_cache(endpoint: str):
    # .open() es un metodo de Path que abre el archivo en modo lectura y devuelve un objeto file-like
    # encoding='utf-8' asegura que el archivo se lea con la codificacion correcta, especialmente si contiene caracteres especiales
    with cache_path(endpoint).open(encoding='utf-8') as f:
        # json.load() lee el contenido del archivo JSON y lo convierte en un objeto de Python
        return json.load(f)


# Funcion para guardar datos de un endpoint/api en el cache local
def save_cache(endpoint: str, data) -> None:

    # mkdir(parents=True, exist_ok=True) crea el directorio cache si no existe.
    # parents=True permite crear cualquier directorio padre necesario, exist_ok=True evita errores si el directorio ya existe.
    CACHE_DIR.mkdir(parents=True, exist_ok=True)  # Crea el directorio cache si no existe

    # Abrimos en modo escritura ('w') y en utf-8 para guardar el JSON con la codificación correcta
    with cache_path(endpoint).open('w', encoding='utf-8') as f:

        # ensure_ascii=False permite que los caracteres no ASCII se guarden correctamente en el archivo JSON,
        # en lugar de ser escapados como secuencias Unicode.
        # ident=2 hace que el JSON se guarde con una indentación de 2 espacios, lo que mejora la legibilidad del archivo.
        json.dump(data, f, ensure_ascii=False, indent=2)



print("Setup listo, ya podemos empezar a trabajar con los datos de SpaceX!")        

 

Setup listo, ya podemos empezar a trabajar con los datos de SpaceX!


# 1. Ingesta de datos desde la API

#### 1.1 Exploracion inicial

In [4]:

# Inspección manual de UN lanzamiento

# Intentamos obtener los datos del ultimo lanzamiento de SpaceX desde la API
try:
    # requests.get() hace una solicitud HTTP GET a la URL especificada
    # y devuelve un objeto Response que contiene la respuesta del servidor.
    response = requests.get(f'{BASE_URL}/launches/latest', timeout=10)

    # Lanza un error si la respuesta no es exitosa (status code 4xx o 5xx)
    response.raise_for_status()  

    # Convierte la respuesta JSON en un diccionario de Python
    sample = response.json() 

    # Guardamos el resultado en cache para futuras referencias
    save_cache('latest_launch', sample)  

except requests.RequestException as e:

    print(f"API no disponible: ({e}). Cargando datos del cache local...")    
    # Carga los datos del cache local si la API no está disponible
    sample = load_cache('latest_launch') 

#Recorremos cada clave-valor del diccionario sample e imprimimos su contenido de forma legible
for key, value in sample.items():

    # Obtiene el tipo de dato del valor
    tipo = type(value).__name__  
    # Obtiene una vista previa de los primeros 60 caracteres del valor
    preview = str(value)[:60]  

    # :25 alinea la clave en 25 caracteres, | es un separador visual, :10 alinea el tipo en 10 caracteres
    # y luego se muestra la vista previa
    # Imprime la clave, el tipo de dato y la vista previa en un formato tabla
    print(f'{key:25} | {tipo:10} | {preview}')


fairings                  | NoneType   | None
links                     | dict       | {'patch': {'small': 'https://images2.imgbox.com/eb/d8/D1Yywp
static_fire_date_utc      | NoneType   | None
static_fire_date_unix     | NoneType   | None
net                       | bool       | False
window                    | NoneType   | None
rocket                    | str        | 5e9d0d95eda69973a809d1ec
success                   | bool       | True
failures                  | list       | []
details                   | NoneType   | None
crew                      | list       | ['62dd7196202306255024d13c', '62dd71c9202306255024d13d', '62
ships                     | list       | []
capsules                  | list       | ['617c05591bad2c661a6e2909']
payloads                  | list       | ['62dd73ed202306255024d145']
launchpad                 | str        | 5e9e4502f509094188566f88
flight_number             | int        | 187
name                      | str        | Crew-5
date_utc            

#### 1.2 Funcion de ingesta robusta

In [6]:
# Definimos una funcion reutilizable para descargar endpoints de la API
# endpoint: el nombre del endpoint a descargar (ejemplo: 'launches', 'rockets', etc.)
# max_retries: el numero maximo de intentos para descargar el endpoint antes de rendirse
# backoff: el factor de aumento del tiempo de espera entre reintentos
# (ejemplo: 1.5 significa que el tiempo de espera se multiplicará por 1.5 después de cada intento fallido)

def fetch_endpoint(endpoint: str, max_retries: int = 3, backoff: float = 1.5) -> list:

    # Construimos la URL completa del endpoint a descargar
    url = f'{BASE_URL}/{endpoint}'
    # 
    last_error = None

    for attempt in range(max_retries):
        try:

            r = requests.get(url, timeout=15)

            r.raise_for_status()  # Lanza un error si la respuesta no es exitosa (status code 4xx o 5xx)

            data = r.json()  # Convierte la respuesta JSON en un objeto de Python (lista o diccionario)

            save_cache(endpoint, data)  # Guarda los datos descargados en el cache local

            return data
        except requests.RequestException as e:
            # guardamos el error para reportarlo si se agotan los intentos
            last_error = e

            # backoff **attempt produce esperas 1.0, .5, 2.25, etc. dependiendo del numero de intento
            wait = backoff * attempt

            # Mostramos un mensaje de error con el numero de intento, el error ocurrido y el tiempo de espera antes del siguiente intento
            print(f' [intento {attempt+1}] Error: {e}. Reintentando en {wait:.1f} segundos...')

            # Pausamos la ejecución del programa durante el tiempo de espera calculado antes de intentar nuevamente
            sleep(wait)

    # Si agotan los intentos, probamos la cache local
    try:

        print(f"Agotados los intentos para descargar {endpoint} desde la API. Cargando datos del cache local...")
        #Devolviendo el json guardado en disco
        return load_cache(endpoint)  # Carga los datos del cache local si la API no está disponible 
    
    #Si tampoco hay cache local, reportamos el error final al usuario
    except FileNotFoundError:
        raise RuntimeError(
            f"No se pudo descargar los datos de {endpoint} desde la API ni cargar el cache local. Tras {max_retries} intentos."
            ) from last_error
    

print("Descargando datos de SpaceX...\n")

# Descargamos los datos de lanzamientos utilizando la función fetch_endpoint, que maneja reintentos y cache local
print("Descargando lanzamientos...")
launches_raw = fetch_endpoint('launches')
print(f'Se descargaron {len(launches_raw)} lanzamientos.\n')

# Descargamos los datos de cohetes utilizando la función fetch_endpoint, que maneja reintentos y cache local
print("Descargando rockets...")
rockets_raw = fetch_endpoint('rockets')
print(f'Se descargaron {len(rockets_raw)} cohetes.\n')

# Descargamos los datos de launchpads utilizando la función fetch_endpoint, que maneja reintentos y cache local
print("Descargando launchpads...")
launchpads_raw = fetch_endpoint('launchpads')
print(f'Se descargaron {len(launchpads_raw)} plataformas de lanzamiento.\n')

# Descargamos los datos de payloads utilizando la función fetch_endpoint, que maneja reintentos y cache local
print("Descargando payloads...")
payloads_raw = fetch_endpoint('payloads')
print(f'Se descargaron {len(payloads_raw)} payloads.\n')
 

Descargando datos de SpaceX...

Descargando lanzamientos...
Se descargaron 205 lanzamientos.

Descargando rockets...
Se descargaron 4 cohetes.

Descargando launchpads...
Se descargaron 6 plataformas de lanzamiento.

Descargando payloads...
Se descargaron 225 payloads.



#### 1.3 JSON a DataFrame

In [7]:

# Convertimos listas de diccionarios en DataFrames de pandas para facilitar su manipulación y análisis
# pd.json_normalize() es una función de pandas que convierte una lista de diccionarios anidados en un DataFrame plano.
# sep='_' define el separador usado al crear nombre de columnas anidadas
df_launches = pd.json_normalize(launches_raw, sep='_')

# repetimos el proceso para cada endpoint descargado
df_rockets = pd.json_normalize(rockets_raw, sep='_')
df_launchpads = pd.json_normalize(launchpads_raw, sep='_')
df_payloads = pd.json_normalize(payloads_raw, sep='_')


# shape muestra el numero de filas y columnas del DataFrame, asi como el tipo de dato de cada columna
print("Estructura de los datos descargados:")
print(f"Launches: {df_launches.shape} (filas, columnas)")
print(f"Rockets: {df_rockets.shape} (filas, columnas)")
print(f"Launchpads: {df_launchpads.shape} (filas, columnas)")
print(f"Payloads: {df_payloads.shape} (filas, columnas)")
 

Estructura de los datos descargados:
Launches: (205, 43) (filas, columnas)
Rockets: (4, 56) (filas, columnas)
Launchpads: (6, 15) (filas, columnas)
Payloads: (225, 34) (filas, columnas)


In [8]:
# Inspeccionamos los nombres de columnas generados por json_normalize
# Esto nos ayuda a entender la estructura de los datos y a identificar posibles columnas de interés para nuestro análisis.
print('Columnas en df_launches:')

# Recorremos cada columna del DataFrame df_launches e imprimimos su nombre
for col in df_launches.columns:
    # Imprime el nombre de la columna con un guion y un espacio al inicio para mejorar la legibilidad
    print(f'- {col}')
 

Columnas en df_launches:
- static_fire_date_utc
- static_fire_date_unix
- net
- window
- rocket
- success
- failures
- details
- crew
- ships
- capsules
- payloads
- launchpad
- flight_number
- name
- date_utc
- date_unix
- date_local
- date_precision
- upcoming
- cores
- auto_update
- tbd
- launch_library_id
- id
- fairings_reused
- fairings_recovery_attempt
- fairings_recovered
- fairings_ships
- links_patch_small
- links_patch_large
- links_reddit_campaign
- links_reddit_launch
- links_reddit_media
- links_reddit_recovery
- links_flickr_small
- links_flickr_original
- links_presskit
- links_webcast
- links_youtube_id
- links_article
- links_wikipedia
- fairings


# 2. Exploracion(EDA con intencion)

In [9]:
# Definimos una función para contar valores distintos en una columna de forma segura, manejando posibles errores de tipo
def safe_nunique(col):
    try:
        # Contamos valores distintos ignorando nulos por defecto
        return col.nunique()

    except TypeError:
        # si la columna tiene listas o diccionarios, convertimos cada valor a string antes de contar los distintos
        return col.apply(str).nunique()
    
# Construimos una tabla de auditoría con una fila por columna orignal
nulos = pd.DataFrame({
    # cantidad de valores faltantes (nulos) en cada columna del DataFrame df_launches
    'nulos': df_launches.isna().sum(),

    # porcentaje de valores faltantes (nulos) respecto al total de filas en el DataFrame df_launches
    # multiplicamos por 100 para obtener el porcentaje y redondeamos a 2 decimales para mejorar la legibilidad
    'pct_nulos': (df_launches.isna().sum() / len(df_launches) * 100).round(2),

    # tipo de dato de cada columna convertido a string para mejor legibilidad
    'tipo': df_launches.dtypes.astype(str),    

    # cantidad de valores distintos en cada columna utilizando la función safe_nunique definida anteriormente
    'valores_distintos': df_launches.apply(safe_nunique)

}).sort_values('pct_nulos', ascending=False)  # Ordenamos la tabla de auditoría por porcentaje de nulos de mayor a menor

print(' Columnas con mayor porcentaje de nulos en df_launches:')
print(nulos.head(15))

#linea en blanco
print()

print('Columnas sin nulos en df_launches:')
print(nulos[nulos['nulos'] == 0].head(15))

#linea en blanco
print()


 Columnas con mayor porcentaje de nulos en df_launches:
                           nulos  pct_nulos     tipo  valores_distintos
fairings                     205     100.00  float64                  0
launch_library_id            133      64.88      str                 72
fairings_recovered           120      58.54   object                  2
links_reddit_media           117      57.07      str                 87
links_presskit               114      55.61      str                 91
fairings_reused              112      54.63   object                  2
links_reddit_recovery        110      53.66      str                 40
fairings_recovery_attempt     98      47.80   object                  2
window                        88      42.93  float64                 25
static_fire_date_unix         84      40.98  float64                121
static_fire_date_utc          84      40.98      str                121
details                       71      34.63      str                133
links_ar

#### 2.3 Despues de revisar los nulos, revisamos unicidad

In [15]:
print(f'Duplicados por id: {df_launches['id'].duplicated().sum()}')
print(f'Duplicados por flight number: {df_launches['flight_number'].duplicated().sum()}')

Duplicados por id: 0
Duplicados por flight number: 4
